In [1]:
import os

from dask.config import config

In [2]:
%pwd

'C:\\Users\\Lenovo\\image classification\\image-classification-\\research'

In [3]:
os.chdir("../")

In [67]:
%pwd

'C:\\Users\\Lenovo\\image classification\\image-classification-'

In [74]:
from dataclasses import dataclass
from pathlib import  Path

@dataclass(frozen=True)
class PrepareCallbacksConfig:
    root_dir: Path
    tensorboard_root_log_dir: Path
    checkpoint_model_filepath: Path

In [75]:
from cnnclassifier.constants import *
from cnnclassifier.utils.common import read_yaml , create_directories

In [76]:
class ConfigurationManager :
    def __init__(
    self ,
    config_filepath = Config_File_path  ,
    params_filepath = Paramas_File_path) :

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])

    def get_prepare_callback_config(self) -> PrepareCallbacksConfig:    ### raed YAML and return

        config = self.config.prepare_callbacks
        model_ckpt_dir = os.path.dirname(config.checkpoint_model_filepath)

        create_directories ([
            Path(model_ckpt_dir) ,
            Path(config.tensorboard_root_log_dir) ,

        ])

        prepare_callbacks_config = PrepareCallbacksConfig(
            root_dir = Path(config.root_dir),
            tensorboard_root_log_dir = Path(config.tensorboard_root_log_dir),
            checkpoint_model_filepath = Path(config.checkpoint_model_filepath),

        )

        return prepare_callbacks_config

In [77]:
import  os
import  urllib.request as  request
from zipfile import  ZipFile
import  tensorflow as tf
import  time

In [78]:
class PrepareCallback:
    def __init__(self, config: PrepareCallbacksConfig):
        self.config = config



    @property
    def _create_tb_callbacks(self):
        timestamp = time.strftime("%Y-%m-%d-%H-%M-%S")
        tb_running_log_dir = os.path.join(
            self.config.tensorboard_root_log_dir,
            f"tb_logs_at_{timestamp}",
        )
        return tf.keras.callbacks.TensorBoard(log_dir=tb_running_log_dir)


    @property
    def _create_ckpt_callbacks(self):
        return tf.keras.callbacks.ModelCheckpoint(
            filepath=self.config.checkpoint_model_filepath,
            save_best_only=True
        )

    def get_tb_ckpt_callbacks(self):
        return [
         self._create_tb_callbacks ,
         self._create_ckpt_callbacks
        ]


In [79]:
try:
    config = ConfigurationManager()
    prepare_callbacks_config = config.get_prepare_callback_config()
    prepare_callbacks = PrepareCallback(config=prepare_callbacks_config)
    callback_list = prepare_callbacks.get_tb_ckpt_callbacks()

except Exception as e:
    raise e

[2026-07-23 21:53:53,534 : INFO : common : yaml file : config\config.yaml loaded successfully]
[2026-07-23 21:53:53,538 : INFO : common : yaml file : params.yaml loaded successfully]
[2026-07-23 21:53:53,540 : INFO : common : created directory in  : artifacts]
[2026-07-23 21:53:53,541 : INFO : common : created directory in  : artifacts\prepare_callbacks\checkpoint_dir]
[2026-07-23 21:53:53,543 : INFO : common : created directory in  : artifacts\prepare_callbacks\tensorboard_log_dir]


In [80]:
import  time
timestamp = time.strftime("%Y-%m-%d-%H-%M-%S")
f"tb_logs_at_{timestamp}"
f"tb_running_log_dir_{timestamp}"

'tb_running_log_dir_2026-07-23-21-56-48'